# Prepare Projected Incidence Data for Calibration Year to Compare to Results of Simulation

In [1]:
from matplotlib.colors import BoundaryNorm, ListedColormap
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

from _configs.country_config import *
from _configs.files_config import *
from _configs.run_config import *

from _utils.utils import *

## Import Data From Rasters

In [2]:
if Path.cwd().name == "population_validation":
    os.chdir(Path.cwd().parent)

print(f"Current Directory: {Path.cwd()}")

districts_raster, _ = read_raster(districts_raster_path)
final_observed_population_raster, metadata = read_raster(observed_population_raster_path)

nodata = metadata["NODATA_value"]

if final_observed_population_raster.shape != districts_raster.shape:
    raise ValueError(
        f"Final population raster and district raster have different shapes: {final_observed_population_raster.shape} vs {districts_raster.shape}"
    )

# Get unique district IDs from full raster, excluding nodata
valid_districts = districts_raster[districts_raster != nodata]
unique_districts = np.unique(valid_districts).astype(int)

print("Unique District IDs:", unique_districts)

# Flatten full rasters
final_observed_population = final_observed_population_raster.flatten().astype(int)
district = districts_raster.flatten().astype(int)

# Keep only cells valid in all three rasters (initial population, final population, and district)
valid_mask = (
    (final_observed_population != nodata) &
    (district != nodata)
)

final_pop_valid = final_observed_population[valid_mask]
district_valid = district[valid_mask]

# Sum population values per district by using district ID as bin index and population as weight (zonal sum via bincount)
sum_final_population_per_district = np.bincount(district_valid, weights=final_pop_valid)

df = pd.DataFrame({
    f"district_id": unique_districts,
    f"observed_population_{observed_population_year}": [sum_final_population_per_district[d] for d in unique_districts],
})

total_final_population = df[f"observed_population_{observed_population_year}"].sum()
print(f"Total Observed Population in {observed_population_year}: {total_final_population:,.0f}")

df

Current Directory: /work/tuv89272/Calibration_Pipeline_v7_NG-SE
Unique District IDs: [ 7  9 11 26 32 35]
Total Observed Population in 2020: 24,572,163


,district_id,observed_population_2020
0,7,6464378.0
1,9,4008484.0
2,11,3284316.0
3,26,2781807.0
4,32,4758921.0
5,35,3274257.0


## Total valid pixels

In [3]:
print("Total valid pixels:", valid_mask.sum())

Total valid pixels: 6851


## Generate Projected Population Raster from observed

In [ ]:
SCALE_FACTOR = TARGET_POPULATION_CALIBRATION_YEAR / df[f"observed_population_{observed_population_year}"].sum()

# Start from full raster
target_year_population_projected_raster = final_observed_population_raster.astype(float).copy()

# Scale only valid cells
valid_mask = (
    (final_observed_population_raster != nodata) &
    (districts_raster != nodata)
)

target_year_population_projected_raster[valid_mask] = final_observed_population_raster[valid_mask] * SCALE_FACTOR

# Keep nodata cells unchanged
target_year_population_projected_raster[~valid_mask] = nodata

write_raster(
    raster=target_year_population_projected_raster,
    file=projected_population_calibration_year_raster_path,
    xllcorner=metadata["xllcorner"],
    yllcorner=metadata["yllcorner"],
    cellsize=metadata["cellsize"],
    mask_raster=districts_raster,  # preserve nodata areas
    nodata=nodata,
)





Scale factor: 1.087265170754402
Projected population in 2024: 26,716,457.000 (target: 26,716,457.000)


In [ ]:
target_population_projected_raster, _ = read_raster(projected_population_calibration_year_raster_path)

# Verify that the sum of the scaled population matches the target
target_population_projected_valid = target_population_projected_raster[valid_mask]
sum_target_population_projected_per_district = np.bincount(district_valid, weights=target_population_projected_valid)

print(f"Scale factor: {SCALE_FACTOR}")
print(f"Projected population in {calibration_year}: {target_population_projected_valid.sum():,.3f} (target: {TARGET_POPULATION_CALIBRATION_YEAR:,.3f})")

## Plot Observed vs Projected Population Map Difference

In [ ]:
observed_pop_raster_path = Path(f"{data_path}/{country_code}_population_{observed_population_year}.asc")
projected_pop_raster_path = Path(f"{data_path}/{country_code}_population_projected_{calibration_year}.asc")

observed_population, _ = read_raster(observed_pop_raster_path)
projected_population, _ = read_raster(projected_pop_raster_path)

# Difference map (Projected - Observed)
population_difference = projected_population - observed_population

# Where sim is nodata but obs exists → sim "contributed 0", so diff = -obs
missing_sim_mask = ~np.isfinite(projected_population) & np.isfinite(observed_population)
population_difference[missing_sim_mask] = -observed_population[missing_sim_mask]

# Where obs is nodata (can't compare) → keep as NaN
population_difference[~np.isfinite(observed_population)] = np.nan

# -----------------------
# Robust linear limits (prevents outliers from crushing contrast)
# -----------------------
combined = np.r_[observed_population[np.isfinite(observed_population)].ravel(), projected_population[np.isfinite(projected_population)].ravel()]

# tweak these if needed
lo, hi = 2, 98
vmin = np.nanpercentile(combined, lo)
vmax = np.nanpercentile(combined, hi)

# -----------------------
# 8-band colormap for Population (Obs/Sim)
# -----------------------
n_bins = 8
bounds = np.r_[-vmax, 0, np.linspace(0, vmax, n_bins)[1:]]

base = plt.get_cmap("viridis", n_bins)        # discretize viridis into 8 colors
bg_white_color = np.array([[1,1,1,1]])
cmap_pop = ListedColormap(np.vstack([bg_white_color, plt.get_cmap("viridis", n_bins)(np.arange(1, n_bins))]))
norm_pop = BoundaryNorm(bounds, cmap_pop.N, clip=True)

# -----------------------
# 8-band diverging for Difference (symmetric, robust)
# -----------------------
dmax = np.nanpercentile(np.abs(population_difference[np.isfinite(population_difference)]), 98)
dbounds = np.linspace(-dmax, dmax, n_bins + 1)

base_d = plt.get_cmap("RdBu_r", n_bins)
cmap_diff = ListedColormap(base_d(np.arange(n_bins)))
norm_diff = BoundaryNorm(dbounds, cmap_diff.N, clip=True)

# -----------------------
# Plot
# -----------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

im0 = axes[0].imshow(observed_population, cmap=cmap_pop, norm=norm_pop, zorder=2)
axes[0].set_title(f"Population Observed in {observed_population_year}")

im1 = axes[1].imshow(projected_population, cmap=cmap_pop, norm=norm_pop, zorder=2)
axes[1].set_title(f"Population Projected in {calibration_year}")

im2 = axes[2].imshow(population_difference, cmap=cmap_diff, norm=norm_diff, zorder=2)
axes[2].set_title(f"Difference in Population (Observed - Projected)")

# Colorbars (banded)
cbar1 = fig.colorbar(im1, ax=axes[:2], shrink=0.85, boundaries=bounds)
cbar1.set_label("Population")
cbar1.set_ticks(bounds)  # show band edges (or use bounds[::2] for fewer ticks)

cbar2 = fig.colorbar(im2, ax=axes[2], shrink=0.85, boundaries=dbounds)
cbar2.set_label("Difference in Population")
cbar2.set_ticks(dbounds)

plt.savefig(f"{data_path}/observed_{observed_population_year}_vs_projected_{calibration_year}_population_pixel_maps.png", dpi=200)
plt.show()

### Write to csv

In [ ]:
df[f'final_population_{calibration_year}_projected'] = df[f"observed_population_{observed_population_year}"] * SCALE_FACTOR

init_pop_sum = df['init_population'].sum()
final_observed_sum = df[f"observed_population_{observed_population_year}"].sum()
final_projected_sum = df[f"observed_population_{calibration_year}_projected"].sum()


print(f'Initial population sum: {init_pop_sum:,.2f}')
print(f'Final population observed sum ({observed_population_year}): {final_observed_sum:,.2f}')
print(f'Final population projected sum ({calibration_year}): {final_projected_sum:,.2f}')

df.to_csv(population_per_district_projected_csv_path, index=False)

print(f"Wrote projected population per district to {population_per_district_projected_csv_path}")

### Scale incidence from population

In [ ]:
pop_df = pd.read_csv(f"{data_path}/population_per_district_projected.csv")
pop_df

In [ ]:
incidence_path =(f"{data_path}/Incidence Data")
# incidence_per_1000 = pd.read_csv(f"{incidence_path}/NG_Incidence_Table.csv")
incidence_per_1000 = pd.read_csv(f"{data_path}/Incidence Data/{country_code}_incidence_data_for_pipeline_DO_NOT_MODIFY.csv")
incidence_per_1000 = incidence_per_1000.dropna()
# incidence_per_1000["Region ID"] = incidence_per_1000["Region ID"].astype(int)
incidence_per_1000

In [ ]:
data_df = pop_df.merge(incidence_per_1000, left_on="district_id", right_on="Region ID", how="left")
data_df

In [ ]:
data_df

In [ ]:
data_df.to_csv(f"{population_incidence_per_district_csv_path}", index=False)

print(f"Wrote population and incidence per district to {population_incidence_per_district_csv_path}")

### Birthrate from initial to calibration year based on initial population and final observed population

In [ ]:
period = calibration_year - initial_year
population_diff = data_df[f'final_population_{calibration_year}_projected'] - data_df['init_population']
change_each_year = population_diff / period

# Build a neat summary table
result_df = data_df[["district_id", "init_population", f'final_population_{calibration_year}_projected']].copy()
result_df["population_diff"] = population_diff
result_df["change_per_year"] = change_each_year

# Formatters for nice printing
formatters = {
    "district_id": lambda x: f"{int(x)}",
    "init_population": lambda x: f"{x:,.0f}",
    f'final_population_{calibration_year}_projected': lambda x: f"{x:,.0f}",
    "population_diff": lambda x: f"{x:,.0f}",
    "change_per_year": lambda x: f"{x:,.2f}",
}

print(f"Population change per district over {period} years (average per year):\n")
print(result_df.to_string(index=False, formatters=formatters))